In [18]:
import lm_eval
import json
import random
import os
import openai
from dotenv import load_dotenv
from datasets import load_dataset
from lm_eval.tasks import TaskManager


In [2]:
load_dotenv()
api_key = os.getenv("LITELLM_ILSP_EVAL_API_KEY")
api_base = os.getenv("LITELLM_HOST")

In [13]:
os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_API_BASE"] = api_base

# Προσθήκη για αντοχή σε network hiccups
os.environ["LITELLM_RETRY_STRATEGY"] = "exponential_backoff"
os.environ["LITELLM_MAX_RETRIES"] = "3"

In [4]:
models_to_test = ["krikri-dpo-latest", "krikri-dpo-context"]

In [5]:
original_dataset = load_dataset("PennyK98/protipa_exams_dataset", split="test")
print(original_dataset)

Dataset({
    features: ['unique_id', 'subject', 'school_level', 'year', 'series', 'label_id', 'question', 'input', 'choices', 'answer', 'answer_index', 'multimodality', 'image_path', 'mark', 'question_type', 'exercise_type', 'source_file'],
    num_rows: 330
})


In [6]:
def filter_dataset(original_dataset):
    filtered_dataset = []
    subjects = ['ΓΛΩΣΣΑ', 'ΜΑΘΗΜΑΤΙΚΑ']
    
    for item in original_dataset:
        # 1. Ελέγχουμε αν το μάθημα είναι μέσα στη λίστα μας
        # 2. Ελέγχουμε αν είναι Multiple Choice
        # 3. Ελέγχουμε αν υπάρχει μόνο μία απάντηση (όχι κόμμα στο answer_index)
        if (item['subject'] in subjects and 
            item['exercise_type'] == 'Multiple Choice' and 
            ',' not in str(item['answer_index'])):
            
            filtered_dataset.append(item)
            
    return filtered_dataset

In [ ]:
#%env LMEVAL_LOG_LEVEL=DEBUG
#!lm_eval \
    #--model hf \
    #--model_args "pretrained=EleutherAI/pythia-2.8b,dtype=float32,device=cpu" \
    #--include_path "C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά" \
    #--tasks greek_protipa_exams \
    #--limit 10

In [7]:
all_filtered = filter_dataset(original_dataset)

# Διαλέγουμε 100 τυχαία δείγματα
pilot_100 = random.sample(all_filtered, min(len(all_filtered), 100))

# ΑΥΤΟΜΑΤΗ ΑΠΟΘΗΚΕΥΣΗ: Δημιουργεί το αρχείο μόνο του κάθε φορά
# Χρησιμοποιούμε απόλυτο path για σιγουριά
base_folder = r'C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πρότυπα_Πειραματικά'
json_full_path = os.path.join(base_folder, 'pilot_data.json')

with open(json_full_path, 'w', encoding='utf-8') as f:
    json.dump(pilot_100, f, ensure_ascii=False, indent=4)

In [8]:
yaml_string = {
    "task": "greek_protipa_exams",
    "dataset_path": "json",
    "dataset_kwargs": {
        "data_files": json_full_path
    },
    "test_split": "train",
    "output_type": "multiple_choice",
    "doc_to_text": "{% if input %}{{input}}\n{% endif %}Ερώτηση: {{question}}\nΑπάντηση:",
    "doc_to_choice": "{{choices}}",
    "doc_to_target": "{{ (answer_index | string).split(',')[0] | int }}",
    "metric_list": [
        {"metric": "acc", "aggregation": "mean", "higher_is_better": True},
        {"metric": "acc_norm", "aggregation": "mean", "higher_is_better": True}
    ]
}

In [ ]:
#Loop Αξιολόγησης και για τα δύο μοντέλα
comparison_results = {}

print (f"Ξεκινάει η αξιολόγηση των μοντέλων: {models_to_test}")

for model_name in models_to_test:
    print (f"Τώρα τρέχει το {model_name} μοντέλο...")
    
    try:
        results = lm_eval.simple_evaluate(
        model="local-chat-completions",
        model_args=f"model={model_name},base_url={api_base},num_fewshot=0,tokenizer=unsloth/llama-3-8b-instruct,max_length=4096,timeout=600",
        tasks=[yaml_string],
        batch_size=1,
        limit=100
        )
        # Αποθηκεύουμε τα αποτελέσματα
        scores = results['results']['greek_protipa_exams']
        comparison_results[model_name] = scores
        print(f"Επιτυχία! Acc: {scores['acc']:.2%}")
        
    except Exception as e:
        print(f"Σφάλμα κατά την αξιολόγηση του μοντέλου {model_name}: {e}")

pretrained=model=krikri-dpo-latest,base_url=http://ec2-3-19-37-251.us-east-2.compute.amazonaws.com:4000/,num_fewshot=0,tokenizer=unsloth/llama-3-8b-instruct,max_length=4096,timeout=600
        appears to be an instruct or chat variant but chat template is not applied. Recommend setting `apply_chat_template` (optionally
        `fewshot_as_multiturn`).


Ξεκινάει η αξιολόγηση των μοντέλων: ['krikri-dpo-latest', 'krikri-dpo-context']
Τώρα τρέχει το krikri-dpo-latest μοντέλο...


pretrained=model=krikri-dpo-context,base_url=http://ec2-3-19-37-251.us-east-2.compute.amazonaws.com:4000/,num_fewshot=0,tokenizer=unsloth/llama-3-8b-instruct,max_length=4096,timeout=600
        appears to be an instruct or chat variant but chat template is not applied. Recommend setting `apply_chat_template` (optionally
        `fewshot_as_multiturn`).


Σφάλμα κατά την αξιολόγηση του μοντέλου krikri-dpo-latest: 'NoneType' object is not iterable
Τώρα τρέχει το krikri-dpo-context μοντέλο...
Σφάλμα κατά την αξιολόγηση του μοντέλου krikri-dpo-context: 'NoneType' object is not iterable


In [ ]:
#Εκτύπωση Τελικού Πίνακα
print("\n" + "="*60)
print(f"{'MODEL':<25} | {'ACCURACY':<10} | {'ACC_NORM':<10}")
print("-" * 60)
for model, metrics in comparison_results.items():
    print(f"{model:<25} | {metrics['acc']:.4f}     | {metrics['acc_norm']:.4f}")
print("="*60)